# Baseline model analysis

In this notebook we do some basic fits in order to determine whether we have enough
baseline data to make a predictor. Since this is a 'back of the envelope' predictor, I will take liberty to drop tons of data in order to quickly get an answer.

In [15]:
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit

In [16]:
delay_data=pd.read_csv('../data/2014-2024data_with_weather.csv')

/var/folders/6d/x65kjw5d1vl7fw7pnbx51yb80000gn/T/ipykernel_47502/2146971062.py:1: DtypeWarning: Columns (0: Line) have mixed types. Specify dtype option on import or set low_memory=False.
  delay_data=pd.read_csv('../data/2014-2024data_with_weather.csv')


We have a warning because the 'Line' column has mixed types (str,int, and float). We fix this by turning all the lines into strings.

In [17]:
# I want to correct the 'Line' column so that each line appears as a string.
# First we need to correct the lines that are appearing as floats (so it has .0 at the end)

def remove_all_after(data_point: str, pivot: str) -> str:
    idx = data_point.find(pivot)
    if idx == -1:
        return data_point
    else:
        return data_point[:idx]

delay_data['Line']=delay_data['Line']\
    .apply(lambda x: remove_all_after(str(x),'.'))

# Similarly, we do the same for the vehicle number.
delay_data['Vehicle']=delay_data['Vehicle'] .apply(lambda x: remove_all_after(str(x),'.'))

# Finally, let me strip the '/B' in the 'Bound' column
delay_data['Bound']=delay_data['Bound']\
    .apply(lambda x: remove_all_after(str(x),'/'))

We drop the 'Incident ID' column as well as any rows with na entries.
Normally one would be more careful about dropping rows, but this is only a back of the
envelope analysis.

In [18]:
try:
    delay_data.drop('Incident ID', axis=1, inplace=True)
except:
    print('Warning: Did you already drop Incident ID?')

print(f'Number of entries before dropping NA: {delay_data.shape[0]}')
delay_data.dropna(inplace=True)
# The vehicle numbers have 'nan' values that are just strings.
delay_data = delay_data[delay_data['Vehicle'] != 'nan']
print(f'Number of entries after dropping NA: {delay_data.shape[0]}')

# So this is a significant amount (>5000) and in our subsequent analysis, we should think
# more critically about how to handle these cases.

Number of entries before dropping NA: 147225
Number of entries after dropping NA: 142189


In [ ]:
# We combine the date and time columns to make a timestamp.
# some of the 'time' entries are missing a seconds column, so we add that first
# by taking the seconds to be :00.
def add_seconds(entry: str) -> str:
    if entry.count(':') < 2:
        return entry +':00'
    else:
        return entry

We now make sure that our dataset is sorted by Datetime to prepare
ourselves for the time series split. We also drop 'Min Gap' since it is highly
correlated with 'Min delay'.

All of the categorical columns need some cleanup. I'm going to drop 'Location' and 'Incident' until
we can get them cleaned up.

I'm also going to drop the sunrise/sunset info because it's currently just a string. We can fix it
up if we want.

In [19]:

try:
    delay_data['Time']=delay_data['Time'].apply(add_seconds)
    delay_data['Datetime']=pd.to_datetime(delay_data['Date']+' '+delay_data['Time'])
    delay_data.drop(columns=['Min Gap', 'Date', 'Time', 'Location', 'Incident', 'sunrise_hhmm',
                             'sunset_hhmm'], inplace=True)
    delay_data.sort_values(by='Datetime', inplace=True)
except:
    print('Warning: Did you already run this cell?')

In [20]:
'''
Let's make sure that the relevant date data are being used.
'''
delay_data['Month'] = delay_data['Datetime'].dt.month.apply(lambda x: str(x))
delay_data['Weekday'] = delay_data['Day']
delay_data['Day'] = delay_data['Datetime'].dt.day.apply(lambda x: str(x))
delay_data['Year'] = delay_data['Datetime'].dt.year.apply(lambda x: str(x))
delay_data['Hour'] = delay_data['Datetime'].dt.hour.apply(lambda x: str(x))
delay_data['Minute'] = delay_data['Datetime'].dt.minute.apply(lambda x: str(x))

try:
    delay_data.drop(['Datetime','date'], axis=1, inplace=True)
except:
    print('Warning: Did you already run this cell?')

In [21]:
delay_data.head()

,Day,Min Delay,Vehicle,Line,Bound,avg_temperature,avg_relative_humidity,avg_dew_point,avg_wind_speed,avg_pressure_station,...,precipitation,rain,snow,daylight,avg_cloud_cover_8,Month,Weekday,Year,Hour,Minute
0,2,4.0,4018,505,E,-17.6,60.0,-23.2,27.0,100.19,...,0.8,0.0,1.0,9.0,7.0,1,Thursday,2014,6,31
1,2,20.0,4128,504,E,-17.6,60.0,-23.2,27.0,100.19,...,0.8,0.0,1.0,9.0,7.0,1,Thursday,2014,12,43
2,2,13.0,4016,501,W,-17.6,60.0,-23.2,27.0,100.19,...,0.8,0.0,1.0,9.0,7.0,1,Thursday,2014,14,1
3,2,7.0,4175,504,W,-17.6,60.0,-23.2,27.0,100.19,...,0.8,0.0,1.0,9.0,7.0,1,Thursday,2014,14,22
4,2,3.0,4080,504,E,-17.6,60.0,-23.2,27.0,100.19,...,0.8,0.0,1.0,9.0,7.0,1,Thursday,2014,16,42


In [22]:
# Let's make a training set and a testing set for our delay data.
# Since our data is ordered by Datetime, we split the training and
# test data by an index.
split_idx=delay_data.shape[0]*4//5
data_train=delay_data.iloc[:split_idx]
data_test=delay_data.iloc[split_idx:]
print(f'Number of entries of training set: {data_train.shape[0]}')
print(f'Number of entries of testing set: {data_test.shape[0]}')

Number of entries of training set: 113751
Number of entries of testing set: 28438


In [23]:
# We can now make our timeseries
n_splits=5
timeseries=TimeSeriesSplit(n_splits=n_splits)
X = data_train.drop('Min Delay', axis=1)
y = data_train['Min Delay']

In [24]:
next(timeseries.split(X))

(array([    0,     1,     2, ..., 18958, 18959, 18960], shape=(18961,)),
 array([18961, 18962, 18963, ..., 37916, 37917, 37918], shape=(18958,)))

In [25]:
# We load a dictionary with the timeseries data.
X_train = dict()
X_test = dict()
y_train = dict()
y_test = dict()
for i, (train_idx, test_idx) in enumerate(timeseries.split(X)):
    X_train[i], X_test[i] = X.iloc[train_idx], X.iloc[test_idx]
    y_train[i], y_test[i] = y.iloc[train_idx], y.iloc[test_idx]

In [26]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import root_mean_squared_error as rmse

# let's get the columns that are numeric vs categorical in order to
# OneHotEncode.
categorical_columns = list(delay_data.select_dtypes(include=str).columns)
numeric_columns = [x for x in delay_data.columns if x not in categorical_columns and x != 'Min Delay']
print(categorical_columns, numeric_columns)

['Day', 'Vehicle', 'Line', 'Bound', 'Month', 'Weekday', 'Year', 'Hour', 'Minute'] ['avg_temperature', 'avg_relative_humidity', 'avg_dew_point', 'avg_wind_speed', 'avg_pressure_station', 'avg_visibility', 'avg_health_index', 'precipitation', 'rain', 'snow', 'daylight', 'avg_cloud_cover_8']


In [27]:
one_hot_transformer = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_columns), #The 'handle_unknown' is doing a *lot* here, due to our data needing cleanup.
    ('datetime', StandardScaler(), numeric_columns)
], remainder='drop')

# List of models in order of flexibility.
models = {
    'dummy': DummyRegressor(strategy='mean'),
    'linear': Pipeline([ ('one hot encode', one_hot_transformer),
                         ('linreg', LinearRegression())]),
    'random_forest': Pipeline([ ('one hot encode', one_hot_transformer),
                         ('random_forest', RandomForestRegressor(
                             n_estimators=500, random_state=37, n_jobs=-1
                         ))
                                ])
}

We should not expect the models to perform well at all since we have particularly periodic data, and our data set is not cleaned up.

In [28]:
score = dict()
for model in models.keys():
    for i in range(n_splits):
        models[model].fit(X_train[i], y_train[i])
        y_pred=models[model].predict(X_test[i])
        score[model,i]=rmse(y_test[i], y_pred)


for model in models.keys():
    score[model] = sum([score[model,i] for i in range(n_splits)])/n_splits
    print(f'For model {model}, the average RMSE is {score[model]}')

For model dummy, the average RMSE is 31.611188265102452
For model linear, the average RMSE is 31.811576213840247
For model random_forest, the average RMSE is 32.763940762643585


This is expectedly pretty awful!